In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from models import NeuralNetBCS
batch_size = 32
seed = 42
lr = 3e-4
epochs = 15
torch.use_deterministic_algorithms(True, warn_only=True)
device = 'mps'

In [ ]:
def val_evaluate(file):
    model = NeuralNetBCS().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.CosineEmbeddingLoss()
    total_loss = 0.0
    total = 0
    for n in range(2, 7):
        val_inputs = np.memmap(f'val_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
        val_targets = np.memmap(f'val_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
        dataset = TensorDataset(torch.from_numpy(val_inputs), torch.from_numpy(val_targets))
        val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        with torch.no_grad():
            for batch_inputs, batch_targets in val_loader:
                batch_inputs = batch_inputs.float().to(device) / 255.0
                batch_targets = batch_targets.float().to(device) * 2 - 1
                inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
                references, query = inputs[:-1], inputs[-1]
                references = torch.stack(references, dim=1)
                emb1, emb2 = model(references, query)
                loss = loss_function(emb1, emb2, batch_targets)
                total_loss += loss.item() * batch_targets.size(0)
                total += batch_targets.size(0)
    print("Validation loss:", total_loss / total)

In [4]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), 
    transforms.ToTensor()
])

In [1]:
def train_model():
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    model = NeuralNetBCS().to(device)
    loss_function = nn.CosineEmbeddingLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for n in range(2, 7):
            print(f'Training model for n={n}, epoch {epoch+1}/{epochs}')
            inputs_mmap = np.memmap(f'train_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(200000, n, 128, 128, 3))
            targets_mmap = np.memmap(f'train_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(200000,))
            dataset = TensorDataset(torch.from_numpy(inputs_mmap), torch.from_numpy(targets_mmap))
            generator = torch.Generator()
            generator.manual_seed(seed)
            train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)
            for batch_inputs, batch_targets in train_loader:
                batch_inputs = batch_inputs.float().to(device) / 255.0
                batch_targets = batch_targets.float().to(device) * 2 - 1
                inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
                inputs = [torch.stack([train_transforms(img) for img in input]).to(device) for input in inputs]
                references, query = inputs[:-1], inputs[-1]
                references = torch.stack(references, dim=1) 
                optimizer.zero_grad()
                emb1, emb2 = model(references, query)
                loss = loss_function(emb1, emb2, batch_targets)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
        print(f'Epoch {epoch+1} avg loss = {running_loss / len(train_loader)}')
        torch.save(model.state_dict(), f'bcs_model/epoch_{epoch+1}.pth')
        val_evaluate(f'bcs_model/epoch_{epoch+1}.pth')

In [ ]:
train_model()

In [ ]:
def calibrate(file):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    model = NeuralNetBCS().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    loss_function = nn.BCELoss()
    X = []
    y = []
    for n in range(2, 7):
        val_inputs = np.memmap(f'val_data/inputs_{n}.dat', dtype=np.uint8, mode='r', shape=(20000, n, 128, 128, 3))
        val_targets = np.memmap(f'val_data/targets_{n}.dat', dtype=np.uint8, mode='r', shape=(20000,))
        dataset = TensorDataset(torch.from_numpy(val_inputs), torch.from_numpy(val_targets))
        val_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        with torch.no_grad():
            for batch_inputs, batch_targets in val_loader:
                batch_inputs = batch_inputs.float().to(device) / 255.0
                batch_targets = batch_targets.float().to(device)
                inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
                references, query = inputs[:-1], inputs[-1]
                references = torch.stack(references, dim=1)
                emb1, emb2 = model(references, query)
                cosine_sim = F.cosine_similarity(emb1, emb2)
                X.extend(cosine_sim.tolist())
                y.extend(batch_targets.tolist())
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1).to(device)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(device)
    calib_dataset = TensorDataset(X, y)
    calib_loader = DataLoader(calib_dataset, batch_size=2048, shuffle=True)
    logistic_regression = nn.Linear(1, 1).to(device)
    optimizer = torch.optim.Adam(logistic_regression.parameters(), lr=0.01)
    loss_function = nn.BCEWithLogitsLoss()
    best_loss, best_weight, best_bias = float('inf'), None, None
    for epoch in range(25):
        total = 0
        loss_sum = 0.0
        for batch_inputs, batch_targets in calib_loader:
            logistic_regression.train()
            optimizer.zero_grad()
            outputs = logistic_regression(batch_inputs)
            loss = loss_function(outputs, batch_targets)
            loss_sum += loss.item() * batch_targets.size(0)
            total += batch_targets.size(0)
            loss.backward()
            optimizer.step()
        if loss_sum / total < best_loss:
            best_loss = loss_sum / total
            best_weight = logistic_regression.weight.item()
            best_bias = logistic_regression.bias.item()
    return best_loss, best_weight, best_bias

In [ ]:
l, w, b = float('inf'), None, None
for i in range(1, 16):
    loss, weight, bias = calibrate(f'bcs_model/epoch_{i}.pth')
    print(f'Epoch {i} calibration loss = {loss}')
    if loss < l:
        l, w, b = loss, weight, bias
    print(f'Epoch {i}: Loss = {l}, Weight = {w}, Bias = {b}')
with open('bcs_model/calibration_params.txt', 'w') as f:
    f.write(f'{w} {b}')

Epoch 1 calibration loss = 0.4404492328262329
Epoch 1: Loss = 0.4404492328262329, Weight = 3.6425716876983643, Bias = -2.2046878337860107
Epoch 2 calibration loss = 0.41731663272857666
Epoch 2: Loss = 0.41731663272857666, Weight = 3.910836696624756, Bias = -2.3173716068267822
Epoch 3 calibration loss = 0.4025454942035675
Epoch 3: Loss = 0.4025454942035675, Weight = 4.009194374084473, Bias = -2.380772352218628
Epoch 4 calibration loss = 0.3985377319908142
Epoch 4: Loss = 0.3985377319908142, Weight = 4.056346416473389, Bias = -2.4278976917266846
Epoch 5 calibration loss = 0.3941749752140045
Epoch 5: Loss = 0.3941749752140045, Weight = 4.075071334838867, Bias = -2.4496331214904785
Epoch 6 calibration loss = 0.3908092057514191
Epoch 6: Loss = 0.3908092057514191, Weight = 4.077301502227783, Bias = -2.436586618423462
Epoch 7 calibration loss = 0.3930233044528961
Epoch 7: Loss = 0.3908092057514191, Weight = 4.077301502227783, Bias = -2.436586618423462
Epoch 8 calibration loss = 0.388883048286